In [4]:
import torch

def check_m4_gpu():
    if torch.backends.mps.is_available():
        print("✅ M4 GPU (Metal) is AVAILABLE via MPS.")
        # Create a small tensor to verify
        device = torch.device("mps")
        x = torch.ones(1, device=device)
        print(f"Test tensor successfully moved to: {x.device}")
    else:
        print("❌ M4 GPU is NOT available to PyTorch. Check your macOS version (12.3+) or PyTorch installation.")

if __name__ == "__main__":
    check_m4_gpu()

✅ M4 GPU (Metal) is AVAILABLE via MPS.
Test tensor successfully moved to: mps:0


In [7]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset" # Adjust if your local path differs
TRAIN_PNG_SOURCE = os.path.join(BASE_PATH, "train/png")
ANNOTATIONS_FILE = os.path.join(BASE_PATH, "train/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# Initialize directories
for c_type in CHART_TYPES:
    os.makedirs(os.path.join(BASE_PATH, f"{c_type}_plots/train/png"), exist_ok=True)

def copy_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(TRAIN_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/train/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_with_m4_cores():
    with open(ANNOTATIONS_FILE, 'r') as f:
        data = json.load(f)

    print(f"Starting split using 10-core parallel processing...")
    counts = {c: 0 for c in CHART_TYPES}
    
    # Using ThreadPoolExecutor to handle I/O bound copying across cores
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\nSplit Summary:")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    split_with_m4_cores()

Starting split using 10-core parallel processing...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157070/157070 [00:28<00:00, 5430.29it/s]



Split Summary:
- vbar_categorical: 52463 images
- hbar_categorical: 52700 images
- dot_line: 26010 images
- line: 25897 images


In [15]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION FOR VAL ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset" 
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# Initialize validation directories
for c_type in CHART_TYPES:
    os.makedirs(os.path.join(BASE_PATH, f"{c_type}_plots/val/png"), exist_ok=True)

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_with_m4():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ Validation annotations not found at: {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"Starting Validation split using 10-core parallel processing...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\nValidation Split Summary:")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    split_val_with_m4()

❌ Validation annotations not found at: /Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset/val/annotations.json


In [18]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = "/Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset"

# PlotQA validation usually sits in its own 'val' subfolder
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

# --- INITIALIZATION ---
def init_val_dirs():
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print("✅ Validation directory structure ready.")

# --- PROCESSING ---
def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type'] # vbar_categorical, hbar_categorical, dot_line, line
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_with_m4():
    # Final check for file existence
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ ERROR: Cannot find validation annotations at: {VAL_ANNOTATIONS}")
        print("Please ensure you have downloaded the Validation Annotations from the official PlotQA links.")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Starting Validation Split (Parallel I/O on 10 Cores)...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_with_m4()

✅ Validation directory structure ready.
❌ ERROR: Cannot find validation annotations at: /Users/akshatchauhan7/Documents/Projects/Spectra/PlotQA-Dataset/val/annotations.json
Please ensure you have downloaded the Validation Annotations from the official PlotQA links.


In [21]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION (Updated from your screenshot) ---
# Your screenshot shows: Documents > Documents > Projects...
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

def init_val_dirs():
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print("✅ Validation specialized directories initialized.")

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_dataset():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ STILL NOT FOUND: {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Processing {len(data)} validation entries using 10 M4 Cores...")
    counts = {c: 0 for c in CHART_TYPES}
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_dataset()

✅ Validation specialized directories initialized.
🚀 Processing 33650 validation entries using 10 M4 Cores...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33650/33650 [00:04<00:00, 6757.63it/s]



--- Validation Split Summary ---
- vbar_categorical: 11240 images
- hbar_categorical: 11292 images
- dot_line: 5571 images
- line: 5547 images


In [1]:
import os
import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# --- CONFIGURATION (Directly from your screenshot) ---
# This absolute path prevents folders from being created in your 'notebooks' section
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
VAL_PNG_SOURCE = os.path.join(BASE_PATH, "val/png") 
VAL_ANNOTATIONS = os.path.join(BASE_PATH, "val/annotations.json")
CHART_TYPES = ['vbar_categorical', 'hbar_categorical', 'dot_line', 'line']

def init_val_dirs():
    # This ensures the new folders are created inside the PlotQA-Dataset directory, not the notebooks folder
    for c_type in CHART_TYPES:
        path = os.path.join(BASE_PATH, f"{c_type}_plots/val/png")
        os.makedirs(path, exist_ok=True)
    print(f"✅ Specialized directories initialized at: {BASE_PATH}")

def copy_val_file(entry):
    img_idx = entry['image_index']
    chart_type = entry['type']
    img_name = f"{img_idx}.png"
    
    src = os.path.join(VAL_PNG_SOURCE, img_name)
    dst = os.path.join(BASE_PATH, f"{chart_type}_plots/val/png", img_name)
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return chart_type
    return None

def split_val_dataset():
    if not os.path.exists(VAL_ANNOTATIONS):
        print(f"❌ ERROR: Cannot find annotations at {VAL_ANNOTATIONS}")
        return

    with open(VAL_ANNOTATIONS, 'r') as f:
        data = json.load(f)

    print(f"🚀 Processing {len(data)} entries from PlotQA Validation set...")
    counts = {c: 0 for c in CHART_TYPES}
    
    # Using your 10-core CPU for maximum I/O speed
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(copy_val_file, data), total=len(data)))

    for r in results:
        if r: counts[r] += 1

    print("\n--- Validation Split Summary ---")
    for c_type, count in counts.items():
        print(f"- {c_type}: {count} images")

if __name__ == "__main__":
    init_val_dirs()
    split_val_dataset()

✅ Specialized directories initialized at: /Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset
🚀 Processing 33650 entries from PlotQA Validation set...


100%|███████████████████████████████████████████████████████████████████| 33650/33650 [00:04<00:00, 7998.51it/s]



--- Validation Split Summary ---
- vbar_categorical: 11240 images
- hbar_categorical: 11292 images
- dot_line: 5571 images
- line: 5547 images
